# AgentCore Infrastructure Setup

This notebook sets up the complete AgentCore infrastructure by:
1. Building Docker-based Lambda deployment packages for all three AgentCore services
2. Initializing and deploying the Terraform infrastructure
3. Verifying the deployment

## Prerequisites

Before running this notebook, ensure you have:
- Docker installed and running
- Terraform installed (version 1.0+)
- AWS CLI configured with appropriate credentials
- Permissions to create IAM roles, Lambda functions, and CloudWatch log groups

## Step 1: Prerequisites Check

Let's verify that all required tools are available and properly configured.

In [1]:
import subprocess
import os
import sys
from pathlib import Path

def _run_subprocess_with_optional_streaming(args, stream_output=False):
    """
    Run subprocess with optional real-time streaming to stdout.
    
    Args:
        args: Command arguments list
        stream_output: Whether to stream output to stdout in real-time
        
    Returns:
        subprocess.CompletedProcess-like object with returncode, stdout, stderr
    """
    if stream_output:
        # Stream output in real-time while capturing for parsing
        print(f"Running command with streaming: {' '.join(args)}")
        
        process = subprocess.Popen(
            args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,  # Merge stderr into stdout for unified streaming
            text=True,
            bufsize=1,  # Line buffered
            universal_newlines=True
        )
        
        stdout_lines = []
        
        # Stream output line by line
        for line in process.stdout:
            # Print to stdout for real-time visibility
            print(line.rstrip())
            sys.stdout.flush()
            
            # Also capture for later parsing
            stdout_lines.append(line)
        
        # Wait for process to complete
        return_code = process.wait()
        
        # Reconstruct stdout and stderr for compatibility
        stdout_content = ''.join(stdout_lines)
        
        # Create a result object similar to subprocess.run return
        class StreamedResult:
            def __init__(self, returncode, stdout, stderr=""):
                self.returncode = returncode
                self.stdout = stdout
                self.stderr = stderr  # Empty since we merged stderr into stdout
        
        return StreamedResult(return_code, stdout_content, "")
        
    else:
        # Use normal subprocess.run for non-streaming mode
        return subprocess.run(args, capture_output=True, text=True)

def run_command(command, description):
    """Run a command and return the result with error handling."""
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            print(f"✅ {description}: OK")
            return True, result.stdout.strip()
        else:
            print(f"❌ {description}: FAILED")
            print(f"   Error: {result.stderr.strip()}")
            return False, result.stderr.strip()
    except subprocess.TimeoutExpired:
        print(f"❌ {description}: TIMEOUT")
        return False, "Command timed out"
    except Exception as e:
        print(f"❌ {description}: ERROR - {str(e)}")
        return False, str(e)

print("🔍 Checking prerequisites...\n")

# Check Docker
docker_ok, docker_version = run_command("docker --version", "Docker installation")
if docker_ok:
    print(f"   Version: {docker_version}")

# Check Docker daemon
if docker_ok:
    docker_running, _ = run_command("docker info > /dev/null 2>&1", "Docker daemon running")

# Check Terraform
terraform_ok, terraform_version = run_command("terraform version", "Terraform installation")
if terraform_ok:
    print(f"   Version: {terraform_version.split()[1]}")

# Check AWS CLI
aws_ok, aws_version = run_command("aws --version", "AWS CLI installation")
if aws_ok:
    print(f"   Version: {aws_version}")

# Check AWS credentials
if aws_ok:
    aws_creds_ok, aws_identity = run_command("aws sts get-caller-identity", "AWS credentials configured")
    if aws_creds_ok:
        import json
        identity = json.loads(aws_identity)
        print(f"   Account: {identity.get('Account', 'Unknown')}")
        print(f"   User/Role: {identity.get('Arn', 'Unknown').split('/')[-1]}")

# Check current directory
current_dir = Path.cwd()
expected_path = "sample-agentic-platform/labs/module6"
if expected_path in str(current_dir):
    print(f"✅ Working directory: OK")
    print(f"   Path: {current_dir}")
else:
    print(f"⚠️  Working directory: Warning - Expected to be in {expected_path}")
    print(f"   Current: {current_dir}")

# Summary
print("\n" + "="*50)
all_good = all([docker_ok, docker_running if docker_ok else False, terraform_ok, aws_ok, aws_creds_ok if aws_ok else False])
if all_good:
    print("🎉 All prerequisites are satisfied! Ready to proceed.")
else:
    print("⚠️  Some prerequisites are missing. Please fix the issues above before continuing.")
print("="*50)

🔍 Checking prerequisites...

✅ Docker installation: OK
   Version: Docker version 27.3.1, build ce1223035a
✅ Docker daemon running: OK
✅ Terraform installation: OK
   Version: v1.11.4
✅ AWS CLI installation: OK
   Version: aws-cli/2.27.62 Python/3.13.4 Darwin/24.5.0 exe/x86_64
✅ AWS credentials configured: OK
   Account: 047719661763
   User/Role: davetbo-Isengard
✅ Working directory: OK
   Path: /Users/davetbo/workplace/cline4/sample-agentic-platform/labs/module6

🎉 All prerequisites are satisfied! Ready to proceed.


## Step 2: Build Lambda Deployment Packages

We'll build three Lambda deployment packages using Docker to ensure consistent, platform-compatible builds:

1. **Memory Provider Lambda** - Handles AgentCore memory operations
2. **MCP Gateway Lambda** - Manages Model Context Protocol gateway functionality  
3. **Runtime Controller Lambda** - Controls AgentCore runtime lifecycle operations

Each package is built using the official AWS Lambda Python 3.13 ARM64 runtime image.

In [2]:
# Change to the agentcore infrastructure directory
agentcore_dir = Path("../../infrastructure/stacks/agentcore")
if not agentcore_dir.exists():
    print(f"❌ AgentCore directory not found: {agentcore_dir.absolute()}")
    print("Please ensure you're running this notebook from the correct location.")
else:
    print(f"✅ Found AgentCore directory: {agentcore_dir.absolute()}")
    os.chdir(agentcore_dir)
    print(f"📁 Changed to: {Path.cwd()}")

✅ Found AgentCore directory: /Users/davetbo/workplace/cline4/sample-agentic-platform/labs/module6/../../infrastructure/stacks/agentcore
📁 Changed to: /Users/davetbo/workplace/cline4/sample-agentic-platform/infrastructure/stacks/agentcore


### 2.1 Build Memory Provider Lambda Package

In [3]:
print("🔨 Building Memory Provider Lambda package...\n")

# Run the memory provider packaging script with streaming output
result = _run_subprocess_with_optional_streaming(
    ["bash", "docker_package_memory_provider_lambda.sh"],
    stream_output=True
)

if result.returncode == 0:
    print("\n✅ Memory Provider Lambda package built successfully!")
    # Check if the zip file was created
    zip_file = Path("agentcore-memory-provider-lambda.zip")
    if zip_file.exists():
        size_mb = zip_file.stat().st_size / (1024 * 1024)
        print(f"📦 Package size: {size_mb:.1f} MB")
    else:
        print("⚠️  Warning: Expected zip file not found")
else:
    print(f"\n❌ Memory Provider Lambda package build failed with return code {result.returncode}")
    sys.exit(1)

🔨 Building Memory Provider Lambda package...

Running command with streaming: bash docker_package_memory_provider_lambda.sh
Packaging Lambda function for AgentCore deployment using Docker...
Package                                                  Repository      Size
Installing:
 amazon-rpm-config-228-9.amzn2023.0.1.noarch             amazonlinux  70.4 kB
 annobin-docs-12.69-1.amzn2023.0.1.noarch                amazonlinux  95.7 kB
 annobin-plugin-gcc-12.69-1.amzn2023.0.1.aarch64         amazonlinux 996.1 kB
 binutils-2.41-50.amzn2023.0.3.aarch64                   amazonlinux   5.8 MB
 cpp-11.5.0-5.amzn2023.0.4.aarch64                       amazonlinux  10.7 MB
 cracklib-2.9.6-27.amzn2023.0.2.aarch64                  amazonlinux  84.7 kB
 cryptsetup-libs-2.6.1-1.amzn2023.0.1.aarch64            amazonlinux 497.1 kB
 dbus-1:1.12.28-1.amzn2023.0.1.aarch64                   amazonlinux   8.7 kB
 dbus-broker-32-1.amzn2023.0.2.aarch64                   amazonlinux 172.5 kB
 dbus-common-1:1.

### 2.2 Build MCP Gateway Lambda Package

In [4]:
print("🔨 Building MCP Gateway Lambda package...\n")

# Run the MCP gateway packaging script with streaming output
result = _run_subprocess_with_optional_streaming(
    ["bash", "docker_package_mcp_gateway.sh"],
    stream_output=True
)

if result.returncode == 0:
    print("\n✅ MCP Gateway Lambda package built successfully!")
    # Check if the zip file was created
    zip_file = Path("agentcore-mcp-gateway-lambda.zip")
    if zip_file.exists():
        size_mb = zip_file.stat().st_size / (1024 * 1024)
        print(f"📦 Package size: {size_mb:.1f} MB")
    else:
        print("⚠️  Warning: Expected zip file not found")
else:
    print(f"\n❌ MCP Gateway Lambda package build failed with return code {result.returncode}")
    sys.exit(1)

🔨 Building MCP Gateway Lambda package...

Running command with streaming: bash docker_package_mcp_gateway.sh
Packaging Lambda function for AgentCore MCP Gateway deployment using Docker...
Package                                                  Repository      Size
Installing:
 amazon-rpm-config-228-9.amzn2023.0.1.noarch             amazonlinux  70.4 kB
 annobin-docs-12.69-1.amzn2023.0.1.noarch                amazonlinux  95.7 kB
 annobin-plugin-gcc-12.69-1.amzn2023.0.1.aarch64         amazonlinux 996.1 kB
 binutils-2.41-50.amzn2023.0.3.aarch64                   amazonlinux   5.8 MB
 cpp-11.5.0-5.amzn2023.0.4.aarch64                       amazonlinux  10.7 MB
 cracklib-2.9.6-27.amzn2023.0.2.aarch64                  amazonlinux  84.7 kB
 cryptsetup-libs-2.6.1-1.amzn2023.0.1.aarch64            amazonlinux 497.1 kB
 dbus-1:1.12.28-1.amzn2023.0.1.aarch64                   amazonlinux   8.7 kB
 dbus-broker-32-1.amzn2023.0.2.aarch64                   amazonlinux 172.5 kB
 dbus-common-1:1.12.

### 2.3 Build Runtime Controller Lambda Package

In [5]:
print("🔨 Building Runtime Controller Lambda package...\n")

# Run the runtime controller packaging script with streaming output
result = _run_subprocess_with_optional_streaming(
    ["bash", "docker_package_agentcore_runtime_lambda.sh"],
    stream_output=True
)

if result.returncode == 0:
    print("\n✅ Runtime Controller Lambda package built successfully!")
    # Check if the zip file was created
    zip_file = Path("agentcore-runtime-lambda.zip")
    if zip_file.exists():
        size_mb = zip_file.stat().st_size / (1024 * 1024)
        print(f"📦 Package size: {size_mb:.1f} MB")
    else:
        print("⚠️  Warning: Expected zip file not found")
else:
    print(f"\n❌ Runtime Controller Lambda package build failed with return code {result.returncode}")
    sys.exit(1)

🔨 Building Runtime Controller Lambda package...

Running command with streaming: bash docker_package_agentcore_runtime_lambda.sh
Packaging Lambda function for AgentCore Runtime deployment using Docker...
Package                                                  Repository      Size
Installing:
 amazon-rpm-config-228-9.amzn2023.0.1.noarch             amazonlinux  70.4 kB
 annobin-docs-12.69-1.amzn2023.0.1.noarch                amazonlinux  95.7 kB
 annobin-plugin-gcc-12.69-1.amzn2023.0.1.aarch64         amazonlinux 996.1 kB
 binutils-2.41-50.amzn2023.0.3.aarch64                   amazonlinux   5.8 MB
 cpp-11.5.0-5.amzn2023.0.4.aarch64                       amazonlinux  10.7 MB
 cracklib-2.9.6-27.amzn2023.0.2.aarch64                  amazonlinux  84.7 kB
 cryptsetup-libs-2.6.1-1.amzn2023.0.1.aarch64            amazonlinux 497.1 kB
 dbus-1:1.12.28-1.amzn2023.0.1.aarch64                   amazonlinux   8.7 kB
 dbus-broker-32-1.amzn2023.0.2.aarch64                   amazonlinux 172.5 kB
 dbu

### 2.4 Package Build Summary

In [6]:
print("📋 Lambda Package Build Summary\n")
print("=" * 50)

packages = [
    ("Memory Provider", "agentcore-memory-provider-lambda.zip"),
    ("MCP Gateway", "agentcore-mcp-gateway-lambda.zip"),
    ("Runtime Controller", "agentcore-runtime-lambda.zip")
]

total_size = 0
all_packages_exist = True

for name, filename in packages:
    zip_file = Path(filename)
    if zip_file.exists():
        size_mb = zip_file.stat().st_size / (1024 * 1024)
        total_size += size_mb
        print(f"✅ {name:<20} {size_mb:>8.1f} MB")
    else:
        print(f"❌ {name:<20} {'MISSING':>8}")
        all_packages_exist = False

print("=" * 50)
print(f"📦 Total package size: {total_size:.1f} MB")

if all_packages_exist:
    print("\n🎉 All Lambda packages built successfully! Ready for Terraform deployment.")
else:
    print("\n❌ Some packages are missing. Please check the build output above.")
    sys.exit(1)

📋 Lambda Package Build Summary

✅ Memory Provider          20.6 MB
✅ MCP Gateway              19.2 MB
✅ Runtime Controller       31.6 MB
📦 Total package size: 71.4 MB

🎉 All Lambda packages built successfully! Ready for Terraform deployment.


## Step 3: Terraform Infrastructure Deployment

Now we'll initialize and deploy the Terraform infrastructure that creates:
- IAM roles and policies for Lambda functions
- Three Lambda functions for AgentCore services
- CloudWatch log groups for monitoring
- Proper security configurations

### 3.1 Terraform Initialization

In [7]:
print("🚀 Initializing Terraform...\n")

# Run terraform init with streaming output
result = _run_subprocess_with_optional_streaming(
    ["terraform", "init"],
    stream_output=True
)

if result.returncode == 0:
    print("\n✅ Terraform initialized successfully!")
else:
    print(f"\n❌ Terraform initialization failed with return code {result.returncode}")
    sys.exit(1)

🚀 Initializing Terraform...

Running command with streaming: terraform init
Initializing the backend...
Initializing provider plugins...
- Reusing previous version of hashicorp/aws from the dependency lock file
- Reusing previous version of hashicorp/archive from the dependency lock file
- Using previously-installed hashicorp/aws v6.9.0
- Using previously-installed hashicorp/archive v2.7.1

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.

✅ Terraform initialized successfully!


### 3.2 Terraform Plan (Optional Review)

Let's review what Terraform plans to create:

In [8]:
print("📋 Generating Terraform plan...\n")

# Run terraform plan with streaming output
result = _run_subprocess_with_optional_streaming(
    ["terraform", "plan"],
    stream_output=True
)

if result.returncode == 0:
    print("\n✅ Terraform plan generated successfully!")
    
    # Extract resource count from plan output
    lines = result.stdout.split('\n')
    for line in lines:
        if 'to add' in line and 'to change' in line and 'to destroy' in line:
            print(f"📊 {line.strip()}")
            break
else:
    print(f"\n❌ Terraform plan failed with return code {result.returncode}")
    sys.exit(1)

📋 Generating Terraform plan...

Running command with streaming: terraform plan
data.archive_file.cognito_pre_signup_trigger_zip[0]: Reading...
data.archive_file.cognito_pre_signup_trigger_zip[0]: Read complete after 0s [id=f8b30b221bb0d2983f4725b3a92af00e9c6993ac]
data.aws_region.current: Reading...
data.aws_caller_identity.current: Reading...
aws_iam_role.agentcore_memory_lambda_role: Refreshing state... [id=agentcore-agentpath-agentcore-memory-lambda-role]
aws_iam_policy.cognito_pre_signup_lambda_policy[0]: Refreshing state... [id=arn:aws:iam::165361166149:policy/agentcore-agentpath-cognito-pre-signup-lambda-policy]
aws_iam_role.agentcore_runtime_lambda_role: Refreshing state... [id=agentcore-agentpath-agentcore-runtime-lambda-role]
aws_iam_role.cognito_pre_signup_lambda_role[0]: Refreshing state... [id=agentcore-agentpath-cognito-pre-signup-lambda-role]
aws_iam_role.agentcore_gateway_lambda_role: Refreshing state... [id=agentcore-agentpath-agentcore-gateway-lambda-role]
data.aws_reg

SystemExit: 1

/Users/davetbo/workplace/cline4/sample-agentic-platform/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 3.3 Terraform Apply

Now we'll deploy the infrastructure with auto-approval:

In [ ]:
print("🚀 Deploying AgentCore infrastructure...\n")
print("This may take several minutes...\n")

# Run terraform apply with auto-approve and streaming output
result = _run_subprocess_with_optional_streaming(
    ["terraform", "apply", "--auto-approve"],
    stream_output=True
)

if result.returncode == 0:
    print("\n🎉 AgentCore infrastructure deployed successfully!")
    
    # Extract apply summary from output
    lines = result.stdout.split('\n')
    for line in lines:
        if 'Apply complete!' in line:
            print(f"📊 {line.strip()}")
            break
else:
    print(f"\n❌ Terraform apply failed with return code {result.returncode}")
    print("\nPlease check the error output above and resolve any issues.")
    sys.exit(1)

## Step 4: Deployment Verification

Let's verify that all resources were created successfully:

In [ ]:
print("🔍 Verifying deployment...\n")

# Get Terraform outputs
result = subprocess.run(
    ["terraform", "output", "-json"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    import json
    outputs = json.loads(result.stdout)
    
    print("📋 Terraform Outputs:")
    print("=" * 60)
    
    # Display key outputs
    key_outputs = [
        ("AWS Region", "aws_region"),
        ("Environment", "environment"),
        ("Memory Lambda ARN", "bedrock_agentcore_memory_lambda_function_arn"),
        ("Runtime Controller ARN", "bedrock_agentcore_runtime_controller_function_arn"),
        ("Memory Retention Days", "memory_retention_days")
    ]
    
    for display_name, key in key_outputs:
        if key in outputs:
            value = outputs[key]["value"]
            print(f"{display_name:<25}: {value}")
    
    print("=" * 60)
    print("\n✅ All outputs retrieved successfully!")
    
else:
    print(f"❌ Failed to retrieve Terraform outputs: {result.stderr}")

### 4.1 Verify Lambda Functions

In [ ]:
print("🔍 Verifying Lambda functions...\n")

# Get the environment name from Terraform outputs
if 'outputs' in locals() and 'environment' in outputs:
    env_name = outputs['environment']['value']
    
    expected_functions = [
        f"{env_name}-agentcore-memory-setup",
        f"{env_name}-bedrock-agentcore-runtime-controller",
        f"{env_name}-bedrock-agentcore-gateway-controller"
    ]
    
    print(f"Environment: {env_name}")
    print("Expected Lambda functions:")
    
    for func_name in expected_functions:
        # Check if Lambda function exists
        result = subprocess.run(
            ["aws", "lambda", "get-function", "--function-name", func_name],
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0:
            func_info = json.loads(result.stdout)
            state = func_info['Configuration']['State']
            runtime = func_info['Configuration']['Runtime']
            print(f"  ✅ {func_name}")
            print(f"     State: {state}, Runtime: {runtime}")
        else:
            print(f"  ❌ {func_name} - Not found or inaccessible")
            
else:
    print("⚠️  Could not retrieve environment name from Terraform outputs")



### Create a convenience function to look up outputs from the TF deployment

In [ ]:
import sys
import os
sys.path.append('../../infrastructure/stacks/agentcore/')
from get_tf_info import GetTfInfo
tf_info = GetTfInfo().get_tf_info()
%store tf_info


In [ ]:
%store -r tf_info
tf_info

## Step 5: Setup Complete!

🎉 **Congratulations!** Your AgentCore infrastructure has been successfully deployed.

### What was created:

1. **IAM Roles and Policies**
   - Roles for each Lambda function with appropriate permissions
   - Policies for Bedrock AgentCore, CloudWatch, and other AWS services

2. **Lambda Functions**
   - **Memory Provider**: Handles AgentCore memory operations
   - **Runtime Controller**: Manages AgentCore runtime lifecycle
   - **MCP Gateway**: Provides Model Context Protocol functionality

3. **CloudWatch Log Groups**
   - Centralized logging for all Lambda functions
   - 7-day retention policy

### Next Steps:

- Proceed to `01_agentcore_memory_operations.ipynb` to test memory operations
- Use `02_agentcore_mcp_gateway_operations.ipynb` to explore MCP gateway functionality
- Check the AWS Console to view your deployed resources

### Cleanup:

When you're done testing, you can clean up all resources by running:
```bash
terraform destroy --auto-approve
```

In [ ]:
print("🎯 Setup Summary")
print("=" * 50)
print("✅ Docker Lambda packages built")
print("✅ Terraform infrastructure deployed")
print("✅ Lambda functions created and verified")
print("✅ CloudWatch logging configured")
print("=" * 50)
print("\n🚀 AgentCore infrastructure is ready for use!")
print("\n📚 Next: Open 01_agentcore_memory_operations.ipynb to start testing")